# PG-Stack · Step 7b — Fair comparator for the within-dataset ensemble validation (C1)

**Why this notebook.** Step 7 (S4) showed that in C1 the trained model M3 was scored on its *raw* out-of-fold scores pooled
across five separately trained fold models, whereas the ensemble used percentile ranks *within each fold*. With only a few
labelled cancers, the score scales of the five fold models differ, and pooling raw scores penalises M3. A deployed system has
one model, so the fair comparator is M3 ranked within each fold (**M3f**). C2 (one model per held-out centre) and all
within-patient metrics (recall@K) are not affected.

This notebook recomputes every C1 comparison against M3f, plus the development blend and the learning curves with fold-ranked
scores (sensitivity analysis). Same input files as Step 7. Runtime ≈ 20–30 min on a T4 GPU. Output: `step7b_results.zip`.

In [ ]:
import os, sys, glob, time, gc, warnings, json
import numpy as np, pandas as pd, lightgbm as lgb
from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import IsolationForest
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from scipy import stats
warnings.filterwarnings("ignore")

WORK = "/kaggle/working"
COMP_DIR = "/kaggle/input/competitions/isic-2024-challenge"
SEED, SEED_C = 42, 2026
N_BOOT = 1000
N_BOOT_RE = 500          # bootstrap resamples per centre for the random-effects variances (S9)
N_PERM_GPU, N_PERM_CPU = 5000, 1000
FRAC_C1 = [0.01, 0.02, 0.03, 0.05, 0.10, 0.20]; REP_C1 = 10
FRAC_C2 = [0.01, 0.02, 0.05, 0.10]; REP_C2 = 5
FRAC_5B = [0.01, 0.02, 0.05, 0.10]; REP_5B = 10
N_BAGS, NEG_PER_POS, MIN_POS_HOSPITAL = 3, 20, 10
N_BAGS_FULL = 5          # S7 full-data models (same recipe as Step 1)
RUN_S7 = True            # retrains three full-data LightGBM models (~10-15 min)

def locate(name):
    if os.path.exists(f"{WORK}/{name}"): return f"{WORK}/{name}"
    for d in range(1, 5):
        hits = glob.glob("/kaggle/input/" + "*/" * d + name)
        if hits: return hits[0]
    return None
NEED = ["oof_step3.csv", "oof_step4c.csv", "emb_pca32.npy", "emb_dinov2_f16.npy", "pgstack_eval.py",
        "step5b_lowlabel.npz", "step5c_c1.npz", "step5c_c2.npz"]
paths = {n: locate(n) for n in NEED}
print(json.dumps(paths, indent=1))
assert all(paths.values()), f"missing: {[k for k, v in paths.items() if v is None]}"
sys.path.insert(0, os.path.dirname(paths["pgstack_eval.py"]))
from pgstack_eval import *
OUT = lambda f: f"{WORK}/s7b_{f}"

In [ ]:
oof = pd.read_csv(paths["oof_step3.csv"])
df = pd.read_csv(f"{COMP_DIR}/train-metadata.csv", low_memory=False)
df = oof[["isic_id", "fold"]].merge(df, on="isic_id", how="left")
assert (df.isic_id.values == oof.isic_id.values).all()
emb = np.load(paths["emb_pca32.npy"]).astype(np.float32)
assert len(emb) == len(oof)

LEAK = ["lesion_id", "iddx_full", "iddx_1", "iddx_2", "iddx_3", "iddx_4", "iddx_5",
        "mel_mitotic_index", "mel_thick_mm", "tbp_lv_dnn_lesion_confidence"]
NUM_RAW = [c for c in df.columns if (c.startswith("tbp_lv_") or c in ["age_approx", "clin_size_long_diam_mm"])
           and c not in LEAK and pd.api.types.is_numeric_dtype(df[c])]
CAT_COLS = ["sex", "anatom_site_general", "tbp_tile_type", "tbp_lv_location", "tbp_lv_location_simple"]

def add_lesion_features(d):
    d = d.copy(); eps = 1e-6
    d["age_approx"] = pd.to_numeric(d["age_approx"], errors="coerce")
    d["lesion_size_ratio"] = d.tbp_lv_minorAxisMM / (d.clin_size_long_diam_mm + eps)
    d["lesion_shape_index"] = d.tbp_lv_areaMM2 / (d.tbp_lv_perimeterMM ** 2 + eps)
    d["hue_contrast"] = (d.tbp_lv_H - d.tbp_lv_Hext).abs()
    d["luminance_contrast"] = (d.tbp_lv_L - d.tbp_lv_Lext).abs()
    d["lesion_color_difference"] = np.sqrt(d.tbp_lv_deltaA**2 + d.tbp_lv_deltaB**2 + d.tbp_lv_deltaL**2)
    d["border_complexity"] = d.tbp_lv_norm_border + d.tbp_lv_symm_2axis
    d["color_uniformity"] = d.tbp_lv_color_std_mean / (d.tbp_lv_radial_color_std_max + eps)
    d["position_distance_3d"] = np.sqrt(d.tbp_lv_x**2 + d.tbp_lv_y**2 + d.tbp_lv_z**2)
    d["perimeter_to_area_ratio"] = d.tbp_lv_perimeterMM / (d.tbp_lv_areaMM2 + eps)
    d["area_to_perimeter_ratio"] = d.tbp_lv_areaMM2 / (d.tbp_lv_perimeterMM + eps)
    d["lesion_visibility_score"] = d.tbp_lv_deltaLBnorm + d.tbp_lv_norm_color
    d["symmetry_border_consistency"] = d.tbp_lv_symm_2axis * d.tbp_lv_norm_border
    d["color_shape_composite"] = (d.tbp_lv_color_std_mean + d.tbp_lv_area_perim_ratio + d.tbp_lv_symm_2axis) / 3
    d["log_area"] = np.log1p(d.tbp_lv_areaMM2)
    d["normalized_lesion_size"] = d.clin_size_long_diam_mm / (d.age_approx + eps)
    d["mean_hue_difference"] = (d.tbp_lv_H + d.tbp_lv_Hext) / 2
    d["std_dev_contrast"] = np.sqrt((d.tbp_lv_deltaA**2 + d.tbp_lv_deltaB**2 + d.tbp_lv_deltaL**2) / 3)
    return d

ENG = ["lesion_size_ratio", "lesion_shape_index", "hue_contrast", "luminance_contrast", "lesion_color_difference",
       "border_complexity", "color_uniformity", "position_distance_3d", "perimeter_to_area_ratio",
       "area_to_perimeter_ratio", "lesion_visibility_score", "symmetry_border_consistency",
       "color_shape_composite", "log_area", "normalized_lesion_size", "mean_hue_difference", "std_dev_contrast"]
df = add_lesion_features(df)
LESION_NUM = sorted(set(NUM_RAW + ENG))
for c in CAT_COLS: df[c] = df[c].fillna("NA").astype("category")
y, folds, pid = oof.target.values, oof.fold.values, oof.patient_id.values
N_FOLDS = int(folds.max()) + 1
POS_COLS = ["tbp_lv_x", "tbp_lv_y", "tbp_lv_z", "position_distance_3d"]
TAB_COLS = [c for c in LESION_NUM if c not in POS_COLS]        # appearance/shape only, for the tabular graph
print(len(df), "lesions |", len(LESION_NUM), "lesion features")

In [ ]:
df["hospital"] = df["attribution"].fillna("unknown").astype(str)
print(df.groupby("hospital").agg(lesions=("isic_id", "size"), cancers=("target", "sum"),
                                 patients=("patient_id", "nunique")).sort_values("cancers", ascending=False))

In [ ]:
g = df.groupby("patient_id"); GLOBAL = []
for c in LESION_NUM:
    df[f"{c}_pnorm"] = (df[c] - g[c].transform("mean")) / (g[c].transform("std") + 1e-6); GLOBAL.append(f"{c}_pnorm")
df["patient_n_lesions"] = g["isic_id"].transform("size"); GLOBAL.append("patient_n_lesions")

In [ ]:
X = df[LESION_NUM].astype(np.float32); X = X.fillna(X.median()).values
tab_idx = [LESION_NUM.index(c) for c in TAB_COLS]
TAB = StandardScaler().fit_transform(X[:, tab_idx]).clip(-6, 6).astype(np.float32)
TAB10 = PCA(10, random_state=SEED).fit_transform(TAB).astype(np.float32)
XYZ = df[["tbp_lv_x", "tbp_lv_y", "tbp_lv_z"]].fillna(0).values.astype(np.float32)
SP = {"tab": TAB, "app": emb, "pos": XYZ}
n = len(df)
GP = {f"gp_{s}_k{k}_{t}": np.full(n, np.nan, np.float32) for s in SP for k in [5, 10, 20] for t in ["dist", "lof"]}
GP["gp_mahal"] = np.full(n, np.nan, np.float32); GP["gp_iforest"] = np.full(n, np.nan, np.float32)

def lof(dist, idx, k):
    d, i = dist[:, :k], idx[:, :k]; reach = np.maximum(d, d[:, -1][i])
    lrd = 1.0 / (reach.mean(1) + 1e-6); return lrd[i].mean(1) / lrd

t0 = time.time()
for j, (p, rows) in enumerate(pd.Series(np.arange(n)).groupby(pid).indices.items()):
    r = np.asarray(rows); m = len(r)
    if m < 6: continue
    km = min(20, m - 1)
    for s, Z in SP.items():
        dist, idx = NearestNeighbors(n_neighbors=km + 1).fit(Z[r]).kneighbors(Z[r]); dist, idx = dist[:, 1:], idx[:, 1:]
        for k in [5, 10, 20]:
            kk = min(k, km); GP[f"gp_{s}_k{k}_dist"][r] = dist[:, :kk].mean(1); GP[f"gp_{s}_k{k}_lof"][r] = lof(dist, idx, kk)
    if m >= 15:
        Zp = TAB10[r]; dz = Zp - Zp.mean(0); C = np.cov(Zp, rowvar=False) + 1e-3 * np.eye(10)
        GP["gp_mahal"][r] = np.sqrt(np.einsum("ij,jk,ik->i", dz, np.linalg.inv(C), dz))
    if m >= 10:
        Zi = np.c_[TAB10[r], emb[r, :10]]
        GP["gp_iforest"][r] = -IsolationForest(n_estimators=100, random_state=SEED).fit(Zi).score_samples(Zi)
    if j % 200 == 0: print(f"  patient {j} | {time.time()-t0:.0f}s")
for k, v in GP.items(): df[k] = v
key = df.patient_id.astype(str) + "|" + df.anatom_site_general.astype(str)
df["gp_pos_k10_dist_siteadj"] = df["gp_pos_k10_dist"] / (df.groupby(key)["gp_pos_k10_dist"].transform("median") + 1e-6)
PRIOR = list(GP.keys()) + ["gp_pos_k10_dist_siteadj"]
print(f"graph prior: {len(PRIOR)} features in {time.time()-t0:.0f}s")

# zero-label score: average percentile rank of the 3 strongest label-free scores
U = np.nanmean(np.stack([pd.Series(df[c]).rank(pct=True).values for c in ["gp_tab_k10_lof", "gp_mahal", "gp_iforest"]]), 0)
U = np.nan_to_num(U, nan=0.5)
print("zero-label U score: AUROC", round(roc_auc_score(y, U), 4), "| pAUC80", round(pauc_80(y, U), 4))

In [ ]:
BASE_F = LESION_NUM + CAT_COLS
FSETS = {"M0_lesion": BASE_F, "M1_global": BASE_F + GLOBAL,
         "M2_prior": BASE_F + PRIOR, "M3_global_prior": BASE_F + GLOBAL + PRIOR}
MODELS = list(FSETS)
XALL = {m: df[f] for m, f in FSETS.items()}
BASE_PARAMS = dict(objective="binary", learning_rate=0.03, num_leaves=63, max_depth=7,
                   feature_fraction=0.6, bagging_fraction=0.8, bagging_freq=1, lambda_l1=0.1, lambda_l2=1.0,
                   n_estimators=300, verbosity=-1)

def fit_predict(model, tr_rows, te_rows, seed):
    pos, neg = tr_rows[y[tr_rows] == 1], tr_rows[y[tr_rows] == 0]
    Xte = XALL[model].iloc[te_rows]; pred = np.zeros(len(te_rows))
    for b in range(N_BAGS):
        rng = np.random.default_rng(seed + b)
        sub = np.r_[pos, rng.choice(neg, min(len(neg), NEG_PER_POS * len(pos)), replace=False)]
        mcs = int(np.clip(len(sub) / 50, 5, 40))
        m = lgb.LGBMClassifier(**BASE_PARAMS, min_child_samples=mcs, random_state=seed + b).fit(XALL[model].iloc[sub], y[sub])
        pred += m.predict_proba(Xte)[:, 1] / N_BAGS
    return pred

pat = df.groupby("patient_id").agg(fold=("fold", "first"), has_pos=("target", "max"),
                                   hospital=("hospital", lambda s: s.mode().iat[0]))
rows_of = pd.Series(np.arange(n)).groupby(pid).indices

def sample_patients(ids_df, share, rng):
    chosen = []
    for hp in [0, 1]:
        ids = ids_df.index[ids_df.has_pos == hp].values
        k = int(round(share * len(ids)))
        if hp == 1: k = max(1, k)                      # at least one cancer patient
        if k > 0: chosen += list(rng.choice(ids, min(k, len(ids)), replace=False))
    return chosen

def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); adj = np.zeros(len(p)); run = 0
    for rank, i in enumerate(o): run = max(run, min(1.0, (len(p) - rank) * p[i])); adj[i] = run
    return adj

In [ ]:
# extra detector for the ablation: Euclidean distance to the patient's centroid in shape/colour space
cent = pd.DataFrame(TAB).groupby(pid).transform("mean").values
df["gp_tab_centroid"] = np.linalg.norm(TAB - cent, axis=1)
codes = pd.factorize(pid)[0]; P = codes.max() + 1
print("patients:", P, "| U reproduced, AUROC =", round(roc_auc_score(y, U), 4))

In [ ]:
_trapz = getattr(np, "trapezoid", None) or getattr(np, "trapz")   # numpy 1.x and 2.x
def prep(yv, s, cv):
    o = np.argsort(s, kind="mergesort"); ss = s[o]
    return (yv[o], np.r_[np.diff(ss) != 0, True], cv[o])
def _roc(pr, cnt=None):
    ys, keep, co = pr; w = np.ones(len(ys)) if cnt is None else cnt[co].astype(float)
    tps = np.r_[0, np.cumsum(w * (ys == 0))[keep]]; fps = np.r_[0, np.cumsum(w * (ys == 1))[keep]]
    return fps / fps[-1], tps / tps[-1]
def pauc_p(pr, cnt=None, min_tpr=0.8):
    fpr, tpr = _roc(pr, cnt); mf = 1 - min_tpr; stop = np.searchsorted(fpr, mf, "right")
    tp = np.append(tpr[:stop], np.interp(mf, fpr[stop-1:stop+1], tpr[stop-1:stop+1])); fp = np.append(fpr[:stop], mf)
    return _trapz(tp, fp)
def auroc_p(pr, cnt=None):
    fpr, tpr = _roc(pr, cnt); return _trapz(tpr, fpr)
def ap_p(pr, cnt=None):
    ys, _, co = pr; ys = ys[::-1]; w = np.ones(len(ys)) if cnt is None else cnt[co[::-1]].astype(float)
    tp = np.cumsum(w * ys); prec = tp / np.maximum(np.cumsum(w), 1e-12)
    return float((prec * w * ys).sum() / max((w * ys).sum(), 1e-12))
_dummy = np.zeros(len(y), int)
def pauc_w(yv, s):  return pauc_p(prep(yv, s, _dummy[:len(yv)]))
def auroc_w(yv, s): return auroc_p(prep(yv, s, _dummy[:len(yv)]))
def ap_w(yv, s):    return ap_p(prep(yv, s, _dummy[:len(yv)]))

rng_b = np.random.default_rng(SEED)
BOOT_C = [np.bincount(rng_b.integers(0, P, P), minlength=P) for _ in range(N_BOOT)]
def boot_dist(metric_p, yv, scores, cv=None, n_boot=None):
    cv = codes if cv is None else cv; preps = [prep(yv, s, cv) for s in scores]
    C = BOOT_C if n_boot is None else BOOT_C[:n_boot]
    return np.array([np.mean([metric_p(pr, c) for pr in preps]) for c in C])
def ci(a): return f"[{np.percentile(a, 2.5):+.4f}, {np.percentile(a, 97.5):+.4f}]"

_s = np.random.default_rng(0).random(len(y)) + y
print("check pAUC:", round(pauc_80(y, _s), 6), round(pauc_w(y, _s), 6), "| AUROC:", round(roc_auc_score(y, _s), 6), round(auroc_w(y, _s), 6))

In [ ]:
o4 = pd.read_csv(paths["oof_step4c.csv"]); o3 = pd.read_csv(paths["oof_step3.csv"])
assert (o4.isic_id.values == df.isic_id.values).all() and (o3.isic_id.values == df.isic_id.values).all()
L5B = dict(np.load(paths["step5b_lowlabel.npz"])); C1 = dict(np.load(paths["step5c_c1.npz"])); C2 = dict(np.load(paths["step5c_c2.npz"]))
HOSP = df.hospital.values
hosp_pos = df.groupby("hospital").target.sum(); HOSPS = sorted(hosp_pos[hosp_pos >= MIN_POS_HOSPITAL].index)

def rank_in(a, groups): return pd.Series(a).groupby(groups).rank(pct=True).values
rU_fold = rank_in(U, folds)
def blend(Uv, Sv, groups, w): return w * rank_in(Uv, groups) + (1 - w) * rank_in(Sv, groups)
def c1(q, r, k=6, Uv=None):
    Uv = U if Uv is None else Uv
    m3 = C1[f"M3|{q}|{r}"]; npos = C1[f"npos|{q}|{r}"]; w = (k / (k + npos))[folds]
    return dict(M3=m3, U=Uv, adaptive=w * rank_in(Uv, folds) + (1 - w) * rank_in(m3, folds))
def c2(h, q, r, k=6, Uv=None):
    Uv = U if Uv is None else Uv
    te = np.where(HOSP == h)[0]; m3 = C2[f"M3|{h}|{q}|{r}"]; n = C2[f"npos|{h}|{q}|{r}"][0]; w = k / (k + n)
    ru = pd.Series(Uv[te]).rank(pct=True).values; rm = pd.Series(m3).rank(pct=True).values
    return te, dict(M3=m3, U=Uv[te], adaptive=w * ru + (1 - w) * rm)
def c2_eval(q, k=6, Uv=None):
    out = []
    for h in HOSPS:
        acc = {m: [] for m in ["M3", "U", "adaptive"]}
        for r in range(REP_C2):
            te, sc = c2(h, q, r, k, Uv)
            for m in acc: acc[m].append(pauc_w(y[te], sc[m]))
        out.append({m: np.mean(v) for m, v in acc.items()})
    return pd.DataFrame(out, index=HOSPS)
print("loaded")

In [ ]:
def topk(s, k):
    r = pd.Series(s).groupby(codes).rank(ascending=False, method="first").values
    f = r <= k; tp = (f & (y == 1)).sum()
    return dict(recall=tp / y.sum(), precision=tp / f.sum(), fp_per_patient=(f & (y == 0)).sum() / P, flagged=f.sum())
def workload(s, frac):
    thr = np.quantile(s, 1 - frac); f = s >= thr
    return (f & (y == 1)).sum() / y.sum(), f.sum() / P

In [ ]:
def prank(v): return pd.Series(v).rank(pct=True).fillna(0.5).values
def wp_auroc(s, yv=None, cv=None):
    yv = y if yv is None else yv; cv = codes if cv is None else cv
    d = pd.DataFrame(dict(c=cv, y=yv, s=s)); num = den = 0.0
    for _, g in d.groupby("c"):
        if 0 < g.y.sum() < len(g):
            num += roc_auc_score(g.y, g.s) * g.y.sum(); den += g.y.sum()
    return num / den
print("U within-patient AUROC:", round(wp_auroc(U), 4))

In [ ]:
import torch
DEV = "cuda" if torch.cuda.is_available() else "cpu"
N_PERM = N_PERM_GPU if DEV == "cuda" else N_PERM_CPU
print("device:", DEV, "| permutations:", N_PERM)
Yt = torch.tensor(y, dtype=torch.float64, device=DEV); CODES_t = torch.tensor(codes, device=DEV)
def pauc_t(s, yt=Yt, mf=0.2):
    o = torch.argsort(s); ss = s[o]; ys = yt[o]
    keep = torch.cat([ss[1:] != ss[:-1], torch.ones(1, dtype=torch.bool, device=s.device)])
    z = torch.zeros(1, dtype=torch.float64, device=s.device)
    tps = torch.cat([z, torch.cumsum(1 - ys, 0)[keep]]); fps = torch.cat([z, torch.cumsum(ys, 0)[keep]])
    fpr, tpr = fps / fps[-1], tps / tps[-1]
    stop = int(torch.searchsorted(fpr, torch.tensor([mf], dtype=torch.float64, device=s.device), right=True)[0])
    x0, x1, y0, y1 = fpr[stop - 1], fpr[stop], tpr[stop - 1], tpr[stop]
    yi = y0 + (y1 - y0) * (mf - x0) / (x1 - x0) if x1 > x0 else y1
    fp = torch.cat([fpr[:stop], torch.tensor([mf], dtype=torch.float64, device=s.device)]); tp = torch.cat([tpr[:stop], yi.reshape(1)])
    return float(torch.trapezoid(tp, fp))
_s = np.random.default_rng(1).random(n) + 0.6 * y
print("check torch vs numpy pAUC:", round(pauc_t(torch.tensor(_s, device=DEV)), 6), round(pauc_w(y, _s), 6))
def perm_test_t(a_list, b_list, n_perm, seed=SEED):
    A = [torch.tensor(pd.Series(a).rank(pct=True).values, device=DEV) for a in a_list]
    B = [torch.tensor(pd.Series(b).rank(pct=True).values, device=DEV) for b in b_list]
    obs = np.mean([pauc_t(a) - pauc_t(b) for a, b in zip(A, B)])
    g = torch.Generator(device=DEV); g.manual_seed(seed); ge = 0
    for _ in range(n_perm):
        sw = (torch.rand(P, generator=g, device=DEV) < 0.5)[CODES_t]
        d = np.mean([pauc_t(torch.where(sw, b, a)) - pauc_t(torch.where(sw, a, b)) for a, b in zip(A, B)])
        ge += d >= obs
    return obs, (1 + ge) / (1 + n_perm)

In [ ]:
N_BOOT_REP = 500
def fr(a): return rank_in(a, folds)            # percentile rank within each test fold
def c1f(q, r, k=6):
    m3 = C1[f"M3|{q}|{r}"]; npos = C1[f"npos|{q}|{r}"]; w = (k / (k + npos))[folds]
    ru, rm = fr(U), fr(m3)
    return dict(M3_raw=m3, M3f=rm, U=U, fixed05=0.5 * ru + 0.5 * rm, adaptive=w * ru + (1 - w) * rm)
def boot_delta(a_list, b_list):
    pa = [prep(y, s, codes) for s in a_list]; pb = [prep(y, s, codes) for s in b_list]
    return np.array([np.mean([pauc_p(x, c) - pauc_p(z, c) for x, z in zip(pa, pb)]) for c in BOOT_C[:N_BOOT_REP]])

## T1 — C1 table with the fold-ranked comparator (M3f), bootstrap CIs and repeats favouring

In [ ]:
rows = []; t0 = time.time()
for q in FRAC_C1:
    sc = [c1f(q, r) for r in range(REP_C1)]
    rec = dict(budget=q, cancers=np.mean([C1[f"npos|{q}|{r}"].mean() for r in range(REP_C1)]))
    for m in ["M3_raw", "M3f", "U", "fixed05", "adaptive"]:
        rec[m] = np.mean([pauc_w(y, s[m]) for s in sc])
    per = [pauc_w(y, s["adaptive"]) - pauc_w(y, s["M3f"]) for s in sc]
    rec["d_vs_M3f"] = rec["adaptive"] - rec["M3f"]; rec["ci_vs_M3f"] = ci(boot_delta([s["adaptive"] for s in sc], [s["M3f"] for s in sc]))
    rec["repeats_favour_vs_M3f"] = int(np.sum(np.array(per) > 0))
    rec["d_vs_U"] = rec["adaptive"] - rec["U"]; rec["ci_vs_U"] = ci(boot_delta([s["adaptive"] for s in sc], [s["U"] for s in sc]))
    rec["d_vs_best"] = rec["adaptive"] - max(rec["M3f"], rec["U"])
    rows.append(rec); print(q, f"{(time.time()-t0)/60:.1f} min")
T1 = pd.DataFrame(rows).round(4); print(T1.to_string(index=False)); T1.to_csv(OUT("T1_C1_fold_ranked.csv"), index=False)

## T2 — Permutation tests (5,000) for adaptive vs M3f at 1, 2 and 3% (prespecified family), plus 5–20% (secondary)

In [ ]:
rows = []; t0 = time.time()
for fam, qs in [("adaptive vs M3f (primary)", [0.01, 0.02, 0.03]), ("adaptive vs M3f (secondary)", [0.05, 0.10, 0.20])]:
    fam_rows = []
    for q in qs:
        sc = [c1f(q, r) for r in range(REP_C1)]
        d, p = perm_test_t([s["adaptive"] for s in sc], [s["M3f"] for s in sc], N_PERM)
        if d < 0:   # test the other direction for budgets where the ensemble is behind
            d2, p2 = perm_test_t([s["M3f"] for s in sc], [s["adaptive"] for s in sc], N_PERM)
            fam_rows.append(dict(family=fam, test=f"C1 @{int(q*100)}%", delta=d, p_one_sided_adaptive_better=p, p_one_sided_M3f_better=p2))
        else:
            fam_rows.append(dict(family=fam, test=f"C1 @{int(q*100)}%", delta=d, p_one_sided_adaptive_better=p, p_one_sided_M3f_better=np.nan))
        print(fam_rows[-1], f"{(time.time()-t0)/60:.1f} min")
    adj = holm([r_["p_one_sided_adaptive_better"] for r_ in fam_rows])
    for r_, a_ in zip(fam_rows, adj): r_["p_holm_adaptive_better"] = a_; r_["n_perm"] = N_PERM
    rows += fam_rows
T2 = pd.DataFrame(rows).round(5); print(T2.to_string(index=False)); T2.to_csv(OUT("T2_perm_adaptive_vs_M3f.csv"), index=False)

## T3 — Sensitivity to *k* in C1 against the fold-ranked comparator

In [ ]:
rows = []
for k in [1, 2, 3, 6, 10, 15, 20, 30]:
    for q in FRAC_C1:
        sc = [c1f(q, r, k) for r in range(REP_C1)]
        a = np.mean([pauc_w(y, s["adaptive"]) for s in sc]); m = np.mean([pauc_w(y, s["M3f"]) for s in sc])
        rows.append(dict(k=k, budget=q, adaptive=a, M3f=m, U=pauc_w(y, U), margin=a - max(m, pauc_w(y, U))))
T3 = pd.DataFrame(rows).round(4)
print(T3.pivot_table(index="k", columns="budget", values="margin").round(4).to_string())
print(T3.groupby("k").margin.agg(["mean", "min", "max"]).round(4).to_string())
T3.to_csv(OUT("T3_k_sensitivity_C1_fold_ranked.csv"), index=False)

## T4 — Clinical metrics in C1 (M3f vs adaptive)

In [ ]:
rows = []
for q in [0.01, 0.02, 0.05, 0.10]:
    sc = [c1f(q, r) for r in range(REP_C1)]
    for m in ["M3_raw", "M3f", "adaptive"]:
        rec = dict(budget=q, model=m, AP=np.mean([ap_w(y, s[m]) for s in sc]),
                   recall5=np.mean([topk(s[m], 5)["recall"] for s in sc]), recall10=np.mean([topk(s[m], 10)["recall"] for s in sc]))
        for f_ in [0.01, 0.02, 0.05]:
            rec[f"sens@{int(f_*100)}%"] = np.mean([workload(s[m], f_)[0] for s in sc])
        rows.append(rec)
T4 = pd.DataFrame(rows).round(4); print(T4.to_string(index=False)); T4.to_csv(OUT("T4_clinical_C1_fold_ranked.csv"), index=False)

## T5 — Development blend and learning curves with fold-ranked scores (sensitivity analysis)

In [ ]:
rows = []
for q in FRAC_5B:
    rec = dict(budget=q)
    for m in ["M0_lesion", "M1_global", "M2_prior", "M3_global_prior"]:
        rec[m + "_raw"] = np.mean([pauc_w(y, L5B[f"{m}|{q}|{r}"]) for r in range(REP_5B)])
        rec[m + "_f"] = np.mean([pauc_w(y, fr(L5B[f"{m}|{q}|{r}"])) for r in range(REP_5B)])
    a = [fr(L5B[f"M3_global_prior|{q}|{r}"]) for r in range(REP_5B)]; b = [fr(L5B[f"M1_global|{q}|{r}"]) for r in range(REP_5B)]
    rec["M3f_minus_M1f"] = rec["M3_global_prior_f"] - rec["M1_global_f"]; rec["ci_M3f_M1f"] = ci(boot_delta(a, b))
    a2 = [fr(L5B[f"M2_prior|{q}|{r}"]) for r in range(REP_5B)]
    rec["M2f_minus_M1f"] = rec["M2_prior_f"] - rec["M1_global_f"]; rec["ci_M2f_M1f"] = ci(boot_delta(a2, b))
    bl = [0.5 * fr(U) + 0.5 * fr(L5B[f"M3_global_prior|{q}|{r}"]) for r in range(REP_5B)]
    rec["fixed05_f"] = np.mean([pauc_w(y, s) for s in bl]); rec["fixed05_minus_M3f"] = rec["fixed05_f"] - rec["M3_global_prior_f"]
    rec["ci_fixed05_M3f"] = ci(boot_delta(bl, a))
    rows.append(rec)
T5 = pd.DataFrame(rows).round(4); print(T5.T.to_string()); T5.to_csv(OUT("T5_learning_curve_fold_ranked.csv"), index=False)

## T6 — Lesion-count tertiles with the fold-ranked comparator

In [ ]:
pc = pd.Series(codes).value_counts().sort_index()                      # lesions per patient
tert_p = pd.qcut(pc.rank(method="first"), 3, labels=["low", "mid", "high"])   # tertiles of patients, as in Step 6
tert = tert_p.reindex(codes).values
rows = []
for t in ["low", "mid", "high"]:
    m = tert == t; rec = dict(tertile=t, cancers=int(y[m].sum()))
    for q in [0.02, 0.10]:
        sc = [c1f(q, r) for r in range(REP_C1)]
        for mm in ["M3_raw", "M3f", "adaptive"]:
            rec[f"{mm}@{int(q*100)}%"] = np.mean([auroc_w(y[m], s[mm][m]) for s in sc])
    rows.append(rec)
T6 = pd.DataFrame(rows).round(4); print(T6.to_string(index=False)); T6.to_csv(OUT("T6_tertiles_fold_ranked.csv"), index=False)

In [ ]:
import zipfile
from IPython.display import FileLink
with zipfile.ZipFile(f"{WORK}/step7b_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(glob.glob(f"{WORK}/s7b_*.csv")): z.write(f, os.path.basename(f))
print(sorted(os.path.basename(f) for f in glob.glob(f"{WORK}/s7b_*.csv")))
FileLink("step7b_results.zip")